## BertMorphTagger issue when text includes encoding errors

If the text happens to include symbols like "�" then the tokenizer ignores that symbol. If that symbol has a separate span in "words" layer (which it usually does) then it will be missing from the Bert's span list and causes errors in layer creation.

In case of BertMorphTagger, the error occurs here:

        assert len(morph_layer) == len(words_layer), \
        f"Failed to rewrite '{morph_layer.name}' layer tokens to '{words_layer.name}' layer words: {len(morph_layer)} != {len(words_layer)}"

To prevent that, the spans that include the special symbol are copied from the "words" layer and then added to moprh layer with empty/None attributes.

This error occurred while tagging balanced_and_reference_corpus.
This tutuorial shows that error and also gives the patch for copying the potentially missing spans.



In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from estnltk import Text

### The code used in BertMorphTagger to solve the missing span issue

```
# collecting spans with encoding problems
probably_missing_spans = []
if "�" in sent_chunk:
    #print("� is in the sentence. Collecting all the spans where that symbol appears.")
    for sp ,span in enumerate(layers[self.words_layer]):
        if "�" in span.text:
            probably_missing_spans.append((span.base_span.start, span.base_span.end))
#print(probably_missing_spans)

# this is for spans with encoding problem: add the spans with None attributes
for span in probably_missing_spans:
    if self.split_pos_form:
        annotation2 = {
                        'bert_tokens': None,
                        'form': "",
                        'partofspeech': "",
                        'probability': None
        }
    else:
        annotation2 = {
            'bert_tokens': None,
            'morph_label': "",
            'probability': None
        }
    morph_layer.add_annotation(span, **annotation2)
    
```

### This is the BertMorphTagger code before the patch

In [4]:
# EstNLTK's BERT-based morphological tagger. Uses Vabamorf's tagset (partofspeech and form tags). 

# Works with next module versions:
# estnltk == 1.7.3
# torch == 2.4.0
# sentencepiece == 0.2.0

# Code references:
# * https://github.com/estnltk/estnltk-model-training/tree/main/morph_tagging
# * https://github.com/estnltk/estnltk/blob/main/estnltk_neural/estnltk_neural/taggers/ner/estbertner_tagger.py
# * https://bitbucket.org/utDigiHum/public/src/8635582194ef1f2dcec53c40abdfa3f29299a067/skriptid/nimeuksuste_m2rgendamine/bert_morph_tagging_tagger.py

import os
import copy
import torch
import collections
import warnings

from typing import MutableMapping, List, Optional

from transformers import AutoConfig, AutoTokenizer, AutoModelForTokenClassification

from estnltk import Text, Layer, Retagger
from estnltk.downloader import get_resource_paths

from estnltk_neural.taggers.embeddings.bert.bert_tokens_to_words_rewriter import BertTokens2WordsRewriter

class BertMorphTagger(Retagger):
    """Applies BERT-based tagging of morphological features (partofspeech and form tags). 
       Uses partofspeech and form tags from Vabamorf's tagset. 
       
       This tagger works either as a tagger (A) or as a retagger (B), depending on 
       whether the flag <code>disambiguate</code> is disabled or enabled. As a Tagger, 
       it creates a new morphological analysis layer, while as a Retagger, it disambiguates 
       an existing morphological analysis layer that has annotations in Vabamorf's tagset. 
       
       A) If <code>disambiguate</code> is False, then the tagger works as a tagger and 
       a new morph layer (<code>output_layer</code>) can be created by calling tagger's 
       tag(...) or make_layer(...) methods. 
       
       B) If <code>disambiguate</code> is True, then <code>output_layer</code> must be set 
       to a morphological analysis layer (which will become an input layer) and which then 
       can be disambiguated by calling tagger's retag(...) or change_layer(...) methods. 
    """

    def __init__(
        self,
        model_location: Optional[str] = None,
        get_top_n_predictions: int = 1,
        output_layer: str = 'bert_morph_tagging',
        sentences_layer: str = 'sentences',
        words_layer: str = 'words',
        token_level: bool = False,
        split_pos_form: bool = True,
        disambiguate: bool = False,
        device: str = "cpu",
        correct_verb_annotation: bool = True,
        change_to_bert_form: bool = False,
        **kwargs
    ):
        """
        Initializes BertMorphTagger

        Args:
            model_location (str, Optional): 
                Full path to the BertMorphTagger's model files directory. If not provided (default), then 
                attempts to use bert_morph_tagging model from estnltk_resources. If that fails (model is 
                missing and downloading fails), then throws an exception.
            get_top_n_predictions (int): Number of labeles predicted for each word.
            output_layer (str): 
                Name of the output morphological annotations layer. Defaults to 'bert_morph_tagging'. 
                Note: if <code>disambiguate==True</code>, then this must be name a Vabamorf-based morph 
                analysis layer which will be disambiguated by calling tagger's <code>retag</code> method. 
            sentences_layer (str): Name of the layer containing sentences.
            words_layer (str): Name of the layer containing words.
            token_level (bool): Whether to tag the text BERT token-level or EstNLTK's word-level. Defaults to False. 
            split_pos_form (bool): Whether to split the predicted labels into two separate features. Defaults to True. 
                <i>Predicted BERT label is a concatenation of Vabamorf's <code>form</code> and <code>partofspeech</code>
                joined with <code>_</code>, for example <code>sg n_S</code></i>.
            disambiguate (bool): Whether the tagger is to be used as an disambiguator of an input morph analyis layer. 
                Defaults to False. If set, then BertMorphTagger can be used to disambiguate an existing Vabamorf-based 
                morph analysis layer by calling <code>BertMorphTagger.retag(text_obj)</code>. Note that the input 
                <code>text_obj</code> must already have <code>output_layer<code> which will be disambiguated. 
            device (str):
                The device to run the bert_morph_tagging model on ('cpu' or 'cuda'). Defaults to 'cpu'.
            correct_verb_annotation (bool):
                If there is word multiplicity but not verb multiplicity and correct annotation could be chosen based on Bert partofspeech, 
                then correct_verb_annotation=True will take the Vabamorf annotation that matches Bert prediction. The comparison is done
                with "neg" removed from Bert predicted "form".
            change_to_bert_form (bool):
                In case of verbs: if there is multiplicity but not verb multiplicity and Bert prediction form contains "neg" while Vabamorf 
                does not, then Vabamorf annotation will be changed to also include "neg" if change_to_bert_form=True. This is tested
                on UD treebank 2.18 where out of 1945 words (with no verb multiplicity) 91.98% matched UD annotation if "neg" was exluded from Bert prediction.

        Raises:
            Exception: Raises when BertMorphTagger's resources have not been downloaded.
        """

        # Configuration parameters
        self.conf_param = ('model_location', 'get_top_n_predictions', 'bert_tokenizer', 'bert_morph_tagging', 'id2label', \
                           'token_level', 'split_pos_form', 'disambiguate', 'sentences_layer', 'words_layer', 'output_layer', \
                           'input_layers', 'output_attributes', 'device', '_bert_tokens_rewriter', 'correct_verb_annotation', 'change_to_bert_form')

        if model_location is None:
            # Try to get the resources path for bert_morph_tagger. Attempt to download, if missing
            resources_path = get_resource_paths("bert_morph_tagging", only_latest=True, download_missing=True)
            if resources_path is None:
                raise Exception( "BertMorphTagger's resources have not been downloaded. "+\
                                 "Use estnltk.download('bert_morph_tagging') to get the missing resources. "+\
                                 "Alternatively, you can specify the directory containing the model "+\
                                 "via parameter model_location at creating the tagger." )
            self.model_location = resources_path

        else:
            self.model_location = model_location

        tokenizer_kwargs = { k:v for (k,v) in kwargs.items() if k in ['do_lower_case', 'use_fast'] }
        self.device = device
        self.get_top_n_predictions = get_top_n_predictions
        self.bert_tokenizer = AutoTokenizer.from_pretrained(self.model_location, **tokenizer_kwargs )
        self.bert_morph_tagging = AutoModelForTokenClassification.from_pretrained(self.model_location,
                                                                        output_attentions = False,
                                                                        output_hidden_states = False)
        self.bert_morph_tagging = self.bert_morph_tagging.to(self.device)
        # Fetch id2label mapping from configuration
        config_dict = AutoConfig.from_pretrained(self.model_location).to_dict()
        self.id2label, _ = config_dict["id2label"], config_dict["label2id"]

        # Set input and output layers
        self.split_pos_form = split_pos_form
        self.disambiguate = disambiguate
        self.token_level = token_level
        self.sentences_layer = sentences_layer
        self.words_layer = words_layer
        self.output_layer = output_layer
        self.input_layers = [sentences_layer, words_layer]
        if self.disambiguate:
            # Check other parameters
            if not self.split_pos_form:
                raise Exception( ('(!) Cannot use BertMorphTagger as a disambiguator (disambiguate==True) if '+\
                                  'split_pos_form==False.') )
            if self.token_level:
                raise Exception( ('(!) Cannot use BertMorphTagger as a disambiguator (disambiguate==True) if '+\
                                  'token_level==True.') )
            if self.get_top_n_predictions > 1:
                warnings.warn( f'(!) Parameter get_top_n_predictions=={self.get_top_n_predictions} has no effect '+\
                                'during retagging/disambiguation (disambiguate==True). Only the label with the '+\
                                'highest probability is used in the disambiguation.' )
            # Add output_layer as an expected input layer that needs to be disambiguated
            self.input_layers.append( self.output_layer )
        self.output_attributes = ['bert_tokens', 'form', 'partofspeech', 'probability'] if self.split_pos_form else ['bert_tokens', 'morph_label', 'probability']
        if self.token_level:
            self._bert_tokens_rewriter = None
        else:
            # Create BertTokens2WordsRewriter for rewriting bert tokens to Estnltk's words
            if self.split_pos_form:
                self._bert_tokens_rewriter = BertTokens2WordsRewriter(
                    bert_tokens_layer=self.output_layer, 
                    input_words_layer=self.words_layer, 
                    output_attributes=self.output_attributes, 
                    output_layer=self.output_layer, 
                    enveloping=False, 
                    ambiguous=True, 
                    decorator=rewriter_decorator_Vabamorf)
            else:
                self._bert_tokens_rewriter = BertTokens2WordsRewriter(
                    bert_tokens_layer=self.output_layer, 
                    input_words_layer=self.words_layer, 
                    output_attributes=self.output_attributes, 
                    output_layer=self.output_layer, 
                    enveloping=False, 
                    ambiguous=True, 
                    decorator=rewriter_decorator_BERT)

        self.correct_verb_annotation = correct_verb_annotation
        self.change_to_bert_form = change_to_bert_form


    def _get_bert_morph_tagging_label_predictions(self, 
                                                  input_str:str, 
                                                  get_top_n_predictions:int = 1):
        """
        Applies Bert on the given input string and returns Bert's tokens,
        token indexes, and top N predicted labels for each token. \n
        Labels will be converted to Vabamorf's annotations type if <code>self.split_pos_form</code> is True.

        Args:
            input_str (str): The input string to be processed.
            top_n (int): Number of top predictions to return for each token.

        Returns:
            List[dict]: Each token's top N predictions with their probabilities.
        """
        # Tokenize the input string
        tokens, batch_encoding = self._tokenize_with_bert(input_str)
        token_indexes = torch.tensor([batch_encoding['input_ids']]).to(self.device)
        # Check if the length exceeds the model's maximum sequence length
        max_seq_length = self.bert_tokenizer.model_max_length
        if token_indexes.size(1) > max_seq_length:
            raise ValueError(f"Input length exceeds the model's max_seq_length of {max_seq_length} tokens")

        # Get predictions
        with torch.no_grad():
            output = self.bert_morph_tagging(token_indexes)

        # Get top N predictions
        top_n_predictions = []
        logits = output.logits.squeeze()  # Shape: [sequence_length, num_labels]
        probs = torch.softmax(logits, dim=-1)  # Convert logits to probabilities

        for i, token_data in enumerate(tokens):
            token_probs = probs[i]  # Probabilities for the current token
            top_n_indices = torch.topk(token_probs, get_top_n_predictions).indices  # Top N label indices
            top_n_labels = [self.id2label[idx.item()] for idx in top_n_indices]  # Convert indices to labels
            top_n_probs = [round(token_probs[idx].item(), 5) for idx in top_n_indices]  # Get probabilities for top N labels

            top_n_predictions.append({
                'token': token_data,
                'predictions': [{'label': label, 'probability': prob} for label, prob in zip(top_n_labels, top_n_probs)]
            })
        # Convert BERT labels to Vabamorf's form and POS
        if self.split_pos_form:
            top_n_predictions = convert_bert_labels_to_vabamorf(top_n_predictions)
        return top_n_predictions

    def _tokenize_with_bert(self, 
                            text:str, 
                            include_spanless:bool=True):

        """
        Tokenizes input string with Bert's tokenizer and returns a list of token spans.
        Each token span is a triple (start, end, token).
        If include_spanless==True (default), then Bert's special "spanless" tokens
        (e.g. [CLS], [SEP]) will also be included with their respective start/end indexes
        set to None.

        Args:
            text(str): The input string to be processed.
            include_spanless(bool): Whether to include Bert's special "spanless" tokens. Defaults to True
        Returns:
            tuple: A tuple containing
            <ul>
                <li>tokens (list): A list of tuples where each tuple contains (start, end, token).</li>
                <li>batch_encoding: The batch encoding object from the BERT tokenizer.</li>
            </ul>
        """
        tokens = []
        #batch_encoding = self.bert_tokenizer(text, return_tensors="pt").to(self.device)
        batch_encoding = self.bert_tokenizer(text)
        for token_id, token in enumerate(batch_encoding.tokens()):
            char_span = batch_encoding.token_to_chars(token_id)
            if char_span is not None:
                tokens.append( (char_span.start, char_span.end, token) )
            elif include_spanless:
                tokens.append( (None, None, token) )
        print("Bert tokens:", tokens, "\n")
        return tokens, batch_encoding


    def _make_layer(self, text: Text, layers: MutableMapping[str, Layer], status: dict) -> Layer:
        """
        Processes the input text to generate a morphological layer by using BERT predictions.

        This method processes each sentence, tokenizes it using BERT, and then assigns morphological
        annotations (i.e., part of speech, form, and probability) to each token. It optionally splits 
        morphological tags into <code>form</code> and <code>partofspeech</code>.

        Args:
            text (Text): The input text object to be processed.
            layers (MutableMapping[str, Layer]): A mapping of layer names to their corresponding layers
                                                in the text object (e.g., sentences, words, etc.).
            status (dict): ... (unused in this function).

        Returns:
            Layer: The morphological layer containing annotations for each token.
        """
        sentences_layer = layers[ self.sentences_layer ]
        words_layer = layers[ self.words_layer ]
        morph_layer = Layer(name=self.output_layer, 
                        attributes=self.output_attributes, 
                        text_object=text, 
                        parent=self.words_layer, 
                        ambiguous=True)

        for k, sentence in enumerate( sentences_layer ):
            sent_start = sentence.start
            sent_text  = sentence.enclosing_text
            # Apply batch processing: split larger input sentence into smaller chunks and process chunk by chunk
            sent_chunks, sent_chunk_indexes = _split_sentence_into_smaller_chunks(sent_text)
            for sent_chunk, (chunk_start, chunk_end) in zip(sent_chunks, sent_chunk_indexes):
                # Get predictions for the sentence
                top_n_predictions = self._get_bert_morph_tagging_label_predictions(sent_chunk, self.get_top_n_predictions) 
                
                # Collect token level annotations (a label for each token)
                for token_data in top_n_predictions:
                    start, end  = token_data['token'][0], token_data['token'][1]
                    bert_tokens = token_data['token'][2]
                    if start is None or end is None:
                        continue  # Ignore sentence start and end tokens (<s>, </s>)
                    all_labels = [pred['label'] for pred in token_data['predictions']]
                    all_probabilities = [pred['probability'] for pred in token_data['predictions']]
                    token_span = (sent_start + chunk_start + start, sent_start + chunk_start + end)

                    for label, prob in zip(all_labels, all_probabilities):
                        if self.split_pos_form:
                            annotation = {
                                'bert_tokens': bert_tokens,
                                'form': label[0],
                                'partofspeech': label[1],
                                'probability': prob
                            }
                        else:
                            annotation = {
                                'bert_tokens': bert_tokens,
                                'morph_label': label,
                                'probability': prob
                            }
                        morph_layer.add_annotation(token_span, **annotation)


        # Add annotations
        if self.token_level:
            # Return token level annotations
            return morph_layer

        else:
            # Aggregate tokens back into words/phrases
            # Use BertTokens2WordsRewriter to convert BERT tokens to words
            # Rewrite to align BERT tokens with words
            morph_layer = self._bert_tokens_rewriter.make_layer(text, layers={morph_layer.name: morph_layer})

        assert len(morph_layer) == len(words_layer), \
        f"Failed to rewrite '{morph_layer.name}' layer tokens to '{words_layer.name}' layer words: {len(morph_layer)} != {len(words_layer)}"

        return morph_layer


    def _change_layer(self, text, layers, status=None):
        # Validate configuration
        if not self.split_pos_form:
            raise Exception( ('(!) Cannot use BertMorphTagger as a disambiguator if '+\
                              'split_pos_form is set False.').format(attr, morph_layer.name) )
        if self.token_level:
            raise Exception( ('(!) Cannot use BertMorphTagger as a disambiguator if '+\
                              'token_level==True.') )
        # Validate inputs
        morph_layer = layers[self.output_layer]
        for attr in ['partofspeech', 'form']:
            if attr not in morph_layer.attributes:
                raise Exception( ('(!) Missing attribute {!r} in output_layer {!r}.'+\
                                  '').format(attr, morph_layer.name) )
        # Create disambiguation layer
        disamb_layer = self._make_layer(text, layers, status)
        # Disambiguate input_morph_analysis_layer
        assert len(morph_layer) == len(disamb_layer)
        for original_word, disamb_word in zip(morph_layer, disamb_layer):
            disamb_pos  = disamb_word.annotations[0]['partofspeech']
            disamb_form = disamb_word.annotations[0]['form']
            # Filter annotations of the original morph layer: keep only those
            # annotations that are matching with the disambiguated annotation
            # (note: there can be multiple suitable annotations due to lemma 
            #  ambiguities)

            if self.correct_verb_annotation:
                # collects pos to check if there is pos multiplicity
                original_pos = [ann['partofspeech'] for ann in original_word.annotations]

            keep_annotations = []
            for annotation in original_word.annotations:
                if annotation['partofspeech'] == disamb_pos and annotation['form'] == disamb_form:
                    # initial strict comparison
                    keep_annotations.append(annotation)
                elif self.correct_verb_annotation and original_pos.count(disamb_pos) == 1 and disamb_pos == "V" and annotation['partofspeech'] == disamb_pos and annotation['form'] == disamb_form.replace("neg", "").strip():
                    # if we get here: we want to choose vabamorf based on Bert verb pos prediction and there is no verb multiplicity in vabamorf, Bert predicted verb, annotation is verb and form matches (with 'neg' removed)
                    if self.change_to_bert_form:
                        # change form to Bert predicted form aka add "neg" to original annotation form 
                        annotation['form'] = disamb_form
                        keep_annotations.append(annotation)
                    else: 
                        # keep original vabamorf annotation
                        keep_annotations.append(annotation)
                    
            if len(keep_annotations) > 0:
                # Only disambiguate if there is at least one annotation left
                # (can't leave a word without any annotations)
                original_word.clear_annotations()
                for annotation in keep_annotations:
                    original_word.add_annotation( annotation )


def convert_bert_labels_to_vabamorf(predictions:List[dict]):
    '''Converts BERT labels into Vabamorf's annotations (<code>partofspeech</code> and <code>form</code>)

    Args:
        predictions (List[dict]): Each token's top N predictions with their probabilities.

    Returns:
        List[dict]: Each token's top N predictions (converted labels) with their probabilities.
    '''
    for prediction in predictions:
        # Update labels by converting to (form, pos)
        for label in prediction['predictions']:
            label_text = label['label']

            if '_' in label_text: # Has both form and pos
                label_split = label_text.split('_')
                form = label_split[0]
                pos = label_split[1]
            else: # Has only form or pos
                if label_text.isupper(): # POS is uppercased
                    form = ''
                    pos = label_text
                else:
                    form = label_text
                    pos = ''

            label['label'] = (form, pos)
    return predictions


def rewriter_decorator_BERT(text_obj, word_index, span):
    """
    Decorator function for <code>BertTokens2WordsRewriter</code>. \n
    Aggregates the <code>morph_labels</code> and <code>probabilities</code> from <code>shared_bert_tokens</code>, finds the most
    common top-1 label, and retrieves the top N labels and their probabilities from the
    first token that contains this top-1 label.

    Args:
        text_obj: EstNLTK Text object.
        words_index: Index of the word in <code>words</code> layer.
        span: EstNLTK's Span object.

    Returns:
        dict: Annotations with the top N labels and probabilities for the word/phrase.
    """

    # Step 1: Find the most frequent top-1 label across all tokens
    top_1_label_counts = collections.Counter()

    for sp in span:
        top_1_label = sp['morph_label'][0] # Get top-1 label
        top_1_label_counts[top_1_label] += 1  # Count occurrences of each top-1 label

    # Identify the most frequent top-1 label
    most_frequent_label = top_1_label_counts.most_common(1)[0][0]
    annotations = list()

    # Step 2: Find the first token that has this most frequent top-1 label
    for sp in span:
        if most_frequent_label in sp['morph_label']:

            # Extract the top N labels and their probabilities starting from this label
            labels = sp['morph_label']
            probabilities = sp['probability']

            assert len(labels) == len(probabilities)

            for (label, prob) in zip(labels, probabilities):
                annotation = {
                'bert_tokens': [sp['bert_tokens'][0] for sp in span],
                'morph_label': label,
                'probability': prob
                }
                annotations.append(annotation)

            # Return the final annotation
            return annotations

    # Fallback if no label found (shouldn't happen)
    raise RuntimeError(f'Could not find a token with this label: {most_frequent_label}')

def rewriter_decorator_Vabamorf(text_obj, word_index, span):
    """
    Decorator function for <code>BertTokens2WordsRewriter</code>. \n
    Aggregates the <code>form</code>, <code>partofspeech</code> and <code>probabilities</code> from <code>shared_bert_tokens</code>, finds the most
    common top-1 label, and retrieves the top N labels and their probabilities from the
    first token that contains this top-1 label.

    Args:
        text_obj: EstNLTK Text object.
        words_index: Index of the word in <code>words</code> layer.
        span: EstNLTK's Span object.

    Returns:
        dict: Annotations with the top N labels and probabilities for the word/phrase.
    """

    # Step 1: Find the most frequent top-1 label across all tokens
    top_1_label_counts = collections.Counter()
    for sp in span:
        forms = sp['form']
        poses = sp['partofspeech']
        for form, pos in zip(forms, poses):
            top_1_label = form + '_' + pos # Get top-1 label
            top_1_label_counts[top_1_label] += 1  # Count occurrences of each top-1 label

    # Identify the most frequent top-1 label
    most_frequent_label = top_1_label_counts.most_common(1)[0][0]
    annotations = list()

    # Step 2: Find the first token that has this most frequent top-1 label
    for sp in span:
        form = most_frequent_label.split('_')[0]
        pos = most_frequent_label.split('_')[1]
        if form in sp['form'] and pos in sp['partofspeech']:

            # Extract the top N labels and their probabilities starting from this label
            tokens = [sp['bert_tokens'][0] for sp in span]
            forms = sp['form']
            poses = sp['partofspeech']
            probabilities = sp['probability']
            for form, pos, probability in zip(forms, poses, probabilities):
                annotation = {
                                'bert_tokens': tokens,
                                'form': form,
                                'partofspeech': pos,
                                'probability': probability
                            }
                annotations.append(annotation)

            # Return the final annotation
            return annotations

    # Fallback if no label found (shouldn't happen)
    raise RuntimeError(f'Could not find a token with this label: {most_frequent_label}')


def _split_sentence_into_smaller_chunks(large_sent: str, max_size:int=900, seek_end_symbols: str='.!?'):
    """
    Splits given large_sent into smaller texts following the text size limit.
    Each smaller text string is allowed to have at most `max_size` characters.
    Returns smaller text strings and their (start, end) indexes in the large_sent.
    """
    assert max_size > 0, f'(!) Invalid batch size: {max_size}'
    if len(large_sent) < max_size:
        return [large_sent], [(0, len(large_sent))]
    chunks = []
    chunk_separators = []
    chunk_indexes = []
    last_chunk_end = 0
    while last_chunk_end < len(large_sent):
        chunk_start = last_chunk_end
        chunk_end = chunk_start + max_size
        if chunk_end >= len(large_sent):
            chunk_end = len(large_sent)
        if isinstance(seek_end_symbols, str):
            # Heuristic: Try to find the last position in the chunk that
            # resembles sentence ending (matches one of the seek_end_symbols)
            i = chunk_end - 1
            while i > chunk_start + 1:
                char = large_sent[i]
                if char in seek_end_symbols:
                    chunk_end = i + 1
                    break
                i -= 1
        chunks.append( large_sent[chunk_start:chunk_end] )
        chunk_indexes.append( (chunk_start, chunk_end) )
        # Find next chunk_start, skip space characters
        updated_chunk_end = chunk_end
        if chunk_end != len(large_sent):
            i = chunk_end
            while i < len(large_sent):
                char = large_sent[i]
                if not char.isspace():
                    updated_chunk_end = i
                    break
                i += 1
            chunk_separators.append( large_sent[chunk_end:updated_chunk_end] )
        last_chunk_end = updated_chunk_end
    assert len(chunk_separators) == len(chunks) - 1
    # Return extracted chunks
    return ( chunks, chunk_indexes )

In [5]:
bert1 = BertMorphTagger(output_layer ="morph_analysis" , disambiguate=True)

In [6]:
t1 = Text("Välistatud ei ole ka see, et järveni ulatus mingi osa Taani hindamisraamatus nimetatud Väo (Uv� tho) külast. ")
t1.tag_layer(["words", "sentences", "morph_analysis"])

Text(text='Välistatud ei ole ka see, et järveni ulatus mingi osa Taani hindamisraamatus nimetatud Väo (Uv� tho) külast. ')

In [7]:
t1["morph_analysis"]

Layer(name='morph_analysis', attributes=('normalized_text', 'lemma', 'root', 'root_tokens', 'ending', 'clitic', 'form', 'partofspeech'), spans=SL[Span('Välistatud', [{'normalized_text': 'Välistatud', 'lemma': 'välistatu', 'root': 'välista=tu', 'root_tokens': ['välistatu'], 'ending': 'd', 'clitic': '', 'form': 'pl n', 'partofspeech': 'S'}]),
Span('ei', [{'normalized_text': 'ei', 'lemma': 'ei', 'root': 'ei', 'root_tokens': ['ei'], 'ending': '0', 'clitic': '', 'form': 'neg', 'partofspeech': 'V'}]),
Span('ole', [{'normalized_text': 'ole', 'lemma': 'olema', 'root': 'ole', 'root_tokens': ['ole'], 'ending': '0', 'clitic': '', 'form': 'o', 'partofspeech': 'V'}]),
Span('ka', [{'normalized_text': 'ka', 'lemma': 'ka', 'root': 'ka', 'root_tokens': ['ka'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'D'}]),
Span('see', [{'normalized_text': 'see', 'lemma': 'see', 'root': 'see', 'root_tokens': ['see'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'P'}]),
Span(',', [{'normalized_text': ',', 'lemma': ',', 'root': ',', 'root_tokens': [','], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}]),
Span('et', [{'normalized_text': 'et', 'lemma': 'et', 'root': 'et', 'root_tokens': ['et'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'J'}]),
Span('järveni', [{'normalized_text': 'järveni', 'lemma': 'järv', 'root': 'järv', 'root_tokens': ['järv'], 'ending': 'ni', 'clitic': '', 'form': 'sg ter', 'partofspeech': 'S'}]),
Span('ulatus', [{'normalized_text': 'ulatus', 'lemma': 'ulatuma', 'root': 'ulatu', 'root_tokens': ['ulatu'], 'ending': 's', 'clitic': '', 'form': 's', 'partofspeech': 'V'}]),
Span('mingi', [{'normalized_text': 'mingi', 'lemma': 'mingi', 'root': 'mingi', 'root_tokens': ['mingi'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'P'}]),
Span('osa', [{'normalized_text': 'osa', 'lemma': 'osa', 'root': 'osa', 'root_tokens': ['osa'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span('Taani', [{'normalized_text': 'Taani', 'lemma': 'Taani', 'root': 'Taani', 'root_tokens': ['Taani'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeech': 'H'}]),
Span('hindamisraamatus', [{'normalized_text': 'hindamisraamatus', 'lemma': 'hindamisraamat', 'root': 'hindamis_raamat', 'root_tokens': ['hindamis', 'raamat'], 'ending': 's', 'clitic': '', 'form': 'sg in', 'partofspeech': 'S'}]),
Span('nimetatud', [{'normalized_text': 'nimetatud', 'lemma': 'nimetatud', 'root': 'nimetatud', 'root_tokens': ['nimetatud'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'A'}, {'normalized_text': 'nimetatud', 'lemma': 'nimetama', 'root': 'nimeta', 'root_tokens': ['nimeta'], 'ending': 'tud', 'clitic': '', 'form': 'tud', 'partofspeech': 'V'}, {'normalized_text': 'nimetatud', 'lemma': 'nimetatud', 'root': 'nimetatud', 'root_tokens': ['nimetatud'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'A'}, {'normalized_text': 'nimetatud', 'lemma': 'nimetatud', 'root': 'nimetatud', 'root_tokens': ['nimetatud'], 'ending': 'd', 'clitic': '', 'form': 'pl n', 'partofspeech': 'A'}]),
Span('Väo', [{'normalized_text': 'Väo', 'lemma': 'Väo', 'root': 'Väo', 'root_tokens': ['Väo'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeech': 'H'}]),
Span('(', [{'normalized_text': '(', 'lemma': '(', 'root': '(', 'root_tokens': ['('], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}]),
Span('Uv', [{'normalized_text': 'Uv', 'lemma': 'Uv', 'root': 'Uv', 'root_tokens': ['Uv'], 'ending': '0', 'clitic': '', 'form': '?', 'partofspeech': 'Y'}]),
Span('�', [{'normalized_text': '�', 'lemma': '�', 'root': '�', 'root_tokens': ['�'], 'ending': '0', 'clitic': '', 'form': '?', 'partofspeech': 'Y'}]),
Span('tho', [{'normalized_text': 'tho', 'lemma': 'tho', 'root': 'tho', 'root_tokens': ['tho'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span(')', [{'normalized_text': ')', 'lemma': ')', 'root': ')', 'root_tokens': [')'], 'ending': '', 'clitic': '', 'form': '', 'partof

The tokenizer excludes the special symbol.

In [8]:
bert1.retag(t1)

Bert tokens: [(None, None, '<s>'), (0, 2, '▁Vä'), (2, 6, 'list'), (6, 10, 'atud'), (10, 13, '▁ei'), (13, 17, '▁ole'), (17, 20, '▁ka'), (20, 24, '▁see'), (24, 25, ','), (25, 28, '▁et'), (28, 33, '▁järv'), (33, 36, 'eni'), (36, 43, '▁ulatus'), (43, 49, '▁mingi'), (49, 53, '▁osa'), (53, 59, '▁Taani'), (59, 68, '▁hindamis'), (68, 76, 'raamatus'), (76, 86, '▁nimetatud'), (86, 89, '▁Vä'), (89, 90, 'o'), (90, 92, '▁('), (92, 93, 'U'), (93, 94, 'v'), (95, 97, '▁t'), (97, 99, 'ho'), (99, 100, ')'), (100, 107, '▁külast'), (107, 108, '.'), (None, None, '</s>')] 



AssertionError: Failed to rewrite 'morph_analysis' layer tokens to 'words' layer words: 21 != 22

## This is an updated code for BertMorphTagger

In [9]:
# EstNLTK's BERT-based morphological tagger. Uses Vabamorf's tagset (partofspeech and form tags). 

# Works with next module versions:
# estnltk == 1.7.3
# torch == 2.4.0
# sentencepiece == 0.2.0

# Code references:
# * https://github.com/estnltk/estnltk-model-training/tree/main/morph_tagging
# * https://github.com/estnltk/estnltk/blob/main/estnltk_neural/estnltk_neural/taggers/ner/estbertner_tagger.py
# * https://bitbucket.org/utDigiHum/public/src/8635582194ef1f2dcec53c40abdfa3f29299a067/skriptid/nimeuksuste_m2rgendamine/bert_morph_tagging_tagger.py

import os
import copy
import torch
import collections
import warnings

from typing import MutableMapping, List, Optional

from transformers import AutoConfig, AutoTokenizer, AutoModelForTokenClassification

from estnltk import Text, Layer, Retagger
from estnltk.downloader import get_resource_paths

from estnltk_neural.taggers.embeddings.bert.bert_tokens_to_words_rewriter import BertTokens2WordsRewriter

class BertMorphTagger2(Retagger):
    """Applies BERT-based tagging of morphological features (partofspeech and form tags). 
       Uses partofspeech and form tags from Vabamorf's tagset. 
       
       This tagger works either as a tagger (A) or as a retagger (B), depending on 
       whether the flag <code>disambiguate</code> is disabled or enabled. As a Tagger, 
       it creates a new morphological analysis layer, while as a Retagger, it disambiguates 
       an existing morphological analysis layer that has annotations in Vabamorf's tagset. 
       
       A) If <code>disambiguate</code> is False, then the tagger works as a tagger and 
       a new morph layer (<code>output_layer</code>) can be created by calling tagger's 
       tag(...) or make_layer(...) methods. 
       
       B) If <code>disambiguate</code> is True, then <code>output_layer</code> must be set 
       to a morphological analysis layer (which will become an input layer) and which then 
       can be disambiguated by calling tagger's retag(...) or change_layer(...) methods. 
    """

    def __init__(
        self,
        model_location: Optional[str] = None,
        get_top_n_predictions: int = 1,
        output_layer: str = 'bert_morph_tagging',
        sentences_layer: str = 'sentences',
        words_layer: str = 'words',
        token_level: bool = False,
        split_pos_form: bool = True,
        disambiguate: bool = False,
        device: str = "cpu",
        correct_verb_annotation: bool = True,
        change_to_bert_form: bool = False,
        **kwargs
    ):
        """
        Initializes BertMorphTagger

        Args:
            model_location (str, Optional): 
                Full path to the BertMorphTagger's model files directory. If not provided (default), then 
                attempts to use bert_morph_tagging model from estnltk_resources. If that fails (model is 
                missing and downloading fails), then throws an exception.
            get_top_n_predictions (int): Number of labeles predicted for each word.
            output_layer (str): 
                Name of the output morphological annotations layer. Defaults to 'bert_morph_tagging'. 
                Note: if <code>disambiguate==True</code>, then this must be name a Vabamorf-based morph 
                analysis layer which will be disambiguated by calling tagger's <code>retag</code> method. 
            sentences_layer (str): Name of the layer containing sentences.
            words_layer (str): Name of the layer containing words.
            token_level (bool): Whether to tag the text BERT token-level or EstNLTK's word-level. Defaults to False. 
            split_pos_form (bool): Whether to split the predicted labels into two separate features. Defaults to True. 
                <i>Predicted BERT label is a concatenation of Vabamorf's <code>form</code> and <code>partofspeech</code>
                joined with <code>_</code>, for example <code>sg n_S</code></i>.
            disambiguate (bool): Whether the tagger is to be used as an disambiguator of an input morph analyis layer. 
                Defaults to False. If set, then BertMorphTagger can be used to disambiguate an existing Vabamorf-based 
                morph analysis layer by calling <code>BertMorphTagger.retag(text_obj)</code>. Note that the input 
                <code>text_obj</code> must already have <code>output_layer<code> which will be disambiguated. 
            device (str):
                The device to run the bert_morph_tagging model on ('cpu' or 'cuda'). Defaults to 'cpu'.
            correct_verb_annotation (bool):
                If there is word multiplicity but not verb multiplicity and correct annotation could be chosen based on Bert partofspeech, 
                then correct_verb_annotation=True will take the Vabamorf annotation that matches Bert prediction. The comparison is done
                with "neg" removed from Bert predicted "form".
            change_to_bert_form (bool):
                In case of verbs: if there is multiplicity but not verb multiplicity and Bert prediction form contains "neg" while Vabamorf 
                does not, then Vabamorf annotation will be changed to also include "neg" if change_to_bert_form=True. This is tested
                on UD treebank 2.18 where out of 1945 words (with no verb multiplicity) 91.98% matched UD annotation if "neg" was exluded from Bert prediction.

        Raises:
            Exception: Raises when BertMorphTagger's resources have not been downloaded.
        """

        # Configuration parameters
        self.conf_param = ('model_location', 'get_top_n_predictions', 'bert_tokenizer', 'bert_morph_tagging', 'id2label', \
                           'token_level', 'split_pos_form', 'disambiguate', 'sentences_layer', 'words_layer', 'output_layer', \
                           'input_layers', 'output_attributes', 'device', '_bert_tokens_rewriter', 'correct_verb_annotation', 'change_to_bert_form')

        if model_location is None:
            # Try to get the resources path for bert_morph_tagger. Attempt to download, if missing
            resources_path = get_resource_paths("bert_morph_tagging", only_latest=True, download_missing=True)
            if resources_path is None:
                raise Exception( "BertMorphTagger's resources have not been downloaded. "+\
                                 "Use estnltk.download('bert_morph_tagging') to get the missing resources. "+\
                                 "Alternatively, you can specify the directory containing the model "+\
                                 "via parameter model_location at creating the tagger." )
            self.model_location = resources_path

        else:
            self.model_location = model_location

        tokenizer_kwargs = { k:v for (k,v) in kwargs.items() if k in ['do_lower_case', 'use_fast'] }
        self.device = device
        self.get_top_n_predictions = get_top_n_predictions
        self.bert_tokenizer = AutoTokenizer.from_pretrained(self.model_location, **tokenizer_kwargs )
        self.bert_morph_tagging = AutoModelForTokenClassification.from_pretrained(self.model_location,
                                                                        output_attentions = False,
                                                                        output_hidden_states = False)
        self.bert_morph_tagging = self.bert_morph_tagging.to(self.device)
        # Fetch id2label mapping from configuration
        config_dict = AutoConfig.from_pretrained(self.model_location).to_dict()
        self.id2label, _ = config_dict["id2label"], config_dict["label2id"]

        # Set input and output layers
        self.split_pos_form = split_pos_form
        self.disambiguate = disambiguate
        self.token_level = token_level
        self.sentences_layer = sentences_layer
        self.words_layer = words_layer
        self.output_layer = output_layer
        self.input_layers = [sentences_layer, words_layer]
        if self.disambiguate:
            # Check other parameters
            if not self.split_pos_form:
                raise Exception( ('(!) Cannot use BertMorphTagger as a disambiguator (disambiguate==True) if '+\
                                  'split_pos_form==False.') )
            if self.token_level:
                raise Exception( ('(!) Cannot use BertMorphTagger as a disambiguator (disambiguate==True) if '+\
                                  'token_level==True.') )
            if self.get_top_n_predictions > 1:
                warnings.warn( f'(!) Parameter get_top_n_predictions=={self.get_top_n_predictions} has no effect '+\
                                'during retagging/disambiguation (disambiguate==True). Only the label with the '+\
                                'highest probability is used in the disambiguation.' )
            # Add output_layer as an expected input layer that needs to be disambiguated
            self.input_layers.append( self.output_layer )
        self.output_attributes = ['bert_tokens', 'form', 'partofspeech', 'probability'] if self.split_pos_form else ['bert_tokens', 'morph_label', 'probability']
        if self.token_level:
            self._bert_tokens_rewriter = None
        else:
            # Create BertTokens2WordsRewriter for rewriting bert tokens to Estnltk's words
            if self.split_pos_form:
                self._bert_tokens_rewriter = BertTokens2WordsRewriter(
                    bert_tokens_layer=self.output_layer, 
                    input_words_layer=self.words_layer, 
                    output_attributes=self.output_attributes, 
                    output_layer=self.output_layer, 
                    enveloping=False, 
                    ambiguous=True, 
                    decorator=rewriter_decorator_Vabamorf)
            else:
                self._bert_tokens_rewriter = BertTokens2WordsRewriter(
                    bert_tokens_layer=self.output_layer, 
                    input_words_layer=self.words_layer, 
                    output_attributes=self.output_attributes, 
                    output_layer=self.output_layer, 
                    enveloping=False, 
                    ambiguous=True, 
                    decorator=rewriter_decorator_BERT)

        self.correct_verb_annotation = correct_verb_annotation
        self.change_to_bert_form = change_to_bert_form


    def _get_bert_morph_tagging_label_predictions(self, 
                                                  input_str:str, 
                                                  get_top_n_predictions:int = 1):
        """
        Applies Bert on the given input string and returns Bert's tokens,
        token indexes, and top N predicted labels for each token. \n
        Labels will be converted to Vabamorf's annotations type if <code>self.split_pos_form</code> is True.

        Args:
            input_str (str): The input string to be processed.
            top_n (int): Number of top predictions to return for each token.

        Returns:
            List[dict]: Each token's top N predictions with their probabilities.
        """
        # Tokenize the input string
        tokens, batch_encoding = self._tokenize_with_bert(input_str)
        token_indexes = torch.tensor([batch_encoding['input_ids']]).to(self.device)
        # Check if the length exceeds the model's maximum sequence length
        max_seq_length = self.bert_tokenizer.model_max_length
        if token_indexes.size(1) > max_seq_length:
            raise ValueError(f"Input length exceeds the model's max_seq_length of {max_seq_length} tokens")

        # Get predictions
        with torch.no_grad():
            output = self.bert_morph_tagging(token_indexes)

        # Get top N predictions
        top_n_predictions = []
        logits = output.logits.squeeze()  # Shape: [sequence_length, num_labels]
        probs = torch.softmax(logits, dim=-1)  # Convert logits to probabilities

        for i, token_data in enumerate(tokens):
            token_probs = probs[i]  # Probabilities for the current token
            top_n_indices = torch.topk(token_probs, get_top_n_predictions).indices  # Top N label indices
            top_n_labels = [self.id2label[idx.item()] for idx in top_n_indices]  # Convert indices to labels
            top_n_probs = [round(token_probs[idx].item(), 5) for idx in top_n_indices]  # Get probabilities for top N labels

            top_n_predictions.append({
                'token': token_data,
                'predictions': [{'label': label, 'probability': prob} for label, prob in zip(top_n_labels, top_n_probs)]
            })
        # Convert BERT labels to Vabamorf's form and POS
        if self.split_pos_form:
            top_n_predictions = convert_bert_labels_to_vabamorf(top_n_predictions)
        return top_n_predictions

    def _tokenize_with_bert(self, 
                            text:str, 
                            include_spanless:bool=True):

        """
        Tokenizes input string with Bert's tokenizer and returns a list of token spans.
        Each token span is a triple (start, end, token).
        If include_spanless==True (default), then Bert's special "spanless" tokens
        (e.g. [CLS], [SEP]) will also be included with their respective start/end indexes
        set to None.

        Args:
            text(str): The input string to be processed.
            include_spanless(bool): Whether to include Bert's special "spanless" tokens. Defaults to True
        Returns:
            tuple: A tuple containing
            <ul>
                <li>tokens (list): A list of tuples where each tuple contains (start, end, token).</li>
                <li>batch_encoding: The batch encoding object from the BERT tokenizer.</li>
            </ul>
        """
        tokens = []
        #batch_encoding = self.bert_tokenizer(text, return_tensors="pt").to(self.device)
        batch_encoding = self.bert_tokenizer(text)
        for token_id, token in enumerate(batch_encoding.tokens()):
            char_span = batch_encoding.token_to_chars(token_id)
            if char_span is not None:
                tokens.append( (char_span.start, char_span.end, token) )
            elif include_spanless:
                tokens.append( (None, None, token) )
        return tokens, batch_encoding


    def _make_layer(self, text: Text, layers: MutableMapping[str, Layer], status: dict) -> Layer:
        """
        Processes the input text to generate a morphological layer by using BERT predictions.

        This method processes each sentence, tokenizes it using BERT, and then assigns morphological
        annotations (i.e., part of speech, form, and probability) to each token. It optionally splits 
        morphological tags into <code>form</code> and <code>partofspeech</code>.

        Args:
            text (Text): The input text object to be processed.
            layers (MutableMapping[str, Layer]): A mapping of layer names to their corresponding layers
                                                in the text object (e.g., sentences, words, etc.).
            status (dict): ... (unused in this function).

        Returns:
            Layer: The morphological layer containing annotations for each token.
        """
        sentences_layer = layers[ self.sentences_layer ]
        words_layer = layers[ self.words_layer ]
        morph_layer = Layer(name=self.output_layer, 
                        attributes=self.output_attributes, 
                        text_object=text, 
                        parent=self.words_layer, 
                        ambiguous=True)

        for k, sentence in enumerate( sentences_layer ):
            sent_start = sentence.start
            sent_text  = sentence.enclosing_text
            # Apply batch processing: split larger input sentence into smaller chunks and process chunk by chunk
            sent_chunks, sent_chunk_indexes = _split_sentence_into_smaller_chunks(sent_text)
            for sent_chunk, (chunk_start, chunk_end) in zip(sent_chunks, sent_chunk_indexes):

                
                ##########################################
                # ADDED PART START
                ##########################################
                # collecting spans with encoding problems
                probably_missing_spans = []
                if "�" in sent_chunk:
                    #print("� is in the sentence. Collecting all the spans where that symbol appears.")
                    for sp ,span in enumerate(layers[self.words_layer]):
                        if "�" in span.text:
                            probably_missing_spans.append((span.base_span.start, span.base_span.end))
                #print(probably_missing_spans)

                # this is for spans with encoding problem: add the spans with None attributes
                for span in probably_missing_spans:
                    if self.split_pos_form:
                        annotation2 = {
                                        'bert_tokens': None,
                                        'form': "",
                                        'partofspeech': "",
                                        'probability': None
                        }
                    else:
                        annotation2 = {
                            'bert_tokens': None,
                            'morph_label': "",
                            'probability': None
                        }
                    morph_layer.add_annotation(span, **annotation2)
                    
                ##########################################
                # ADDED PART END
                ##########################################

                # Get predictions for the sentence
                top_n_predictions = self._get_bert_morph_tagging_label_predictions(sent_chunk, self.get_top_n_predictions) 

                # Collect token level annotations (a label for each token)
                for token_data in top_n_predictions:
                    start, end  = token_data['token'][0], token_data['token'][1]
                    bert_tokens = token_data['token'][2]
                    if start is None or end is None:
                        continue  # Ignore sentence start and end tokens (<s>, </s>)
                    all_labels = [pred['label'] for pred in token_data['predictions']]
                    all_probabilities = [pred['probability'] for pred in token_data['predictions']]
                    token_span = (sent_start + chunk_start + start, sent_start + chunk_start + end)

                    for label, prob in zip(all_labels, all_probabilities):
                        if self.split_pos_form:
                            annotation = {
                                'bert_tokens': bert_tokens,
                                'form': label[0],
                                'partofspeech': label[1],
                                'probability': prob
                            }
                        else:
                            annotation = {
                                'bert_tokens': bert_tokens,
                                'morph_label': label,
                                'probability': prob
                            }
                        morph_layer.add_annotation(token_span, **annotation)


        # Add annotations
        if self.token_level:
            # Return token level annotations
            return morph_layer

        else:
            # Aggregate tokens back into words/phrases
            # Use BertTokens2WordsRewriter to convert BERT tokens to words
            # Rewrite to align BERT tokens with words
            morph_layer = self._bert_tokens_rewriter.make_layer(text, layers={morph_layer.name: morph_layer})

        assert len(morph_layer) == len(words_layer), \
        f"Failed to rewrite '{morph_layer.name}' layer tokens to '{words_layer.name}' layer words: {len(morph_layer)} != {len(words_layer)}"

        return morph_layer


    def _change_layer(self, text, layers, status=None):
        # Validate configuration
        if not self.split_pos_form:
            raise Exception( ('(!) Cannot use BertMorphTagger as a disambiguator if '+\
                              'split_pos_form is set False.').format(attr, morph_layer.name) )
        if self.token_level:
            raise Exception( ('(!) Cannot use BertMorphTagger as a disambiguator if '+\
                              'token_level==True.') )
        # Validate inputs
        morph_layer = layers[self.output_layer]
        for attr in ['partofspeech', 'form']:
            if attr not in morph_layer.attributes:
                raise Exception( ('(!) Missing attribute {!r} in output_layer {!r}.'+\
                                  '').format(attr, morph_layer.name) )
        # Create disambiguation layer
        disamb_layer = self._make_layer(text, layers, status)
        # Disambiguate input_morph_analysis_layer
        assert len(morph_layer) == len(disamb_layer)
        for original_word, disamb_word in zip(morph_layer, disamb_layer):
            disamb_pos  = disamb_word.annotations[0]['partofspeech']
            disamb_form = disamb_word.annotations[0]['form']
            # Filter annotations of the original morph layer: keep only those
            # annotations that are matching with the disambiguated annotation
            # (note: there can be multiple suitable annotations due to lemma 
            #  ambiguities)

            if self.correct_verb_annotation:
                # collects pos to check if there is pos multiplicity
                original_pos = [ann['partofspeech'] for ann in original_word.annotations]

            keep_annotations = []
            for annotation in original_word.annotations:
                if annotation['partofspeech'] == disamb_pos and annotation['form'] == disamb_form:
                    # initial strict comparison
                    keep_annotations.append(annotation)
                elif self.correct_verb_annotation and original_pos.count(disamb_pos) == 1 and disamb_pos == "V" and annotation['partofspeech'] == disamb_pos and annotation['form'] == disamb_form.replace("neg", "").strip():
                    # if we get here: we want to choose vabamorf based on Bert verb pos prediction and there is no verb multiplicity in vabamorf, Bert predicted verb, annotation is verb and form matches (with 'neg' removed)
                    if self.change_to_bert_form:
                        # change form to Bert predicted form aka add "neg" to original annotation form 
                        annotation['form'] = disamb_form
                        keep_annotations.append(annotation)
                    else: 
                        # keep original vabamorf annotation
                        keep_annotations.append(annotation)
                    
            if len(keep_annotations) > 0:
                # Only disambiguate if there is at least one annotation left
                # (can't leave a word without any annotations)
                original_word.clear_annotations()
                for annotation in keep_annotations:
                    original_word.add_annotation( annotation )


def convert_bert_labels_to_vabamorf(predictions:List[dict]):
    '''Converts BERT labels into Vabamorf's annotations (<code>partofspeech</code> and <code>form</code>)

    Args:
        predictions (List[dict]): Each token's top N predictions with their probabilities.

    Returns:
        List[dict]: Each token's top N predictions (converted labels) with their probabilities.
    '''
    for prediction in predictions:
        # Update labels by converting to (form, pos)
        for label in prediction['predictions']:
            label_text = label['label']

            if '_' in label_text: # Has both form and pos
                label_split = label_text.split('_')
                form = label_split[0]
                pos = label_split[1]
            else: # Has only form or pos
                if label_text.isupper(): # POS is uppercased
                    form = ''
                    pos = label_text
                else:
                    form = label_text
                    pos = ''

            label['label'] = (form, pos)
    return predictions


def rewriter_decorator_BERT(text_obj, word_index, span):
    """
    Decorator function for <code>BertTokens2WordsRewriter</code>. \n
    Aggregates the <code>morph_labels</code> and <code>probabilities</code> from <code>shared_bert_tokens</code>, finds the most
    common top-1 label, and retrieves the top N labels and their probabilities from the
    first token that contains this top-1 label.

    Args:
        text_obj: EstNLTK Text object.
        words_index: Index of the word in <code>words</code> layer.
        span: EstNLTK's Span object.

    Returns:
        dict: Annotations with the top N labels and probabilities for the word/phrase.
    """

    # Step 1: Find the most frequent top-1 label across all tokens
    top_1_label_counts = collections.Counter()

    for sp in span:
        top_1_label = sp['morph_label'][0] # Get top-1 label
        top_1_label_counts[top_1_label] += 1  # Count occurrences of each top-1 label

    # Identify the most frequent top-1 label
    most_frequent_label = top_1_label_counts.most_common(1)[0][0]
    annotations = list()

    # Step 2: Find the first token that has this most frequent top-1 label
    for sp in span:
        if most_frequent_label in sp['morph_label']:

            # Extract the top N labels and their probabilities starting from this label
            labels = sp['morph_label']
            probabilities = sp['probability']

            assert len(labels) == len(probabilities)

            for (label, prob) in zip(labels, probabilities):
                annotation = {
                'bert_tokens': [sp['bert_tokens'][0] for sp in span],
                'morph_label': label,
                'probability': prob
                }
                annotations.append(annotation)

            # Return the final annotation
            return annotations

    # Fallback if no label found (shouldn't happen)
    raise RuntimeError(f'Could not find a token with this label: {most_frequent_label}')

def rewriter_decorator_Vabamorf(text_obj, word_index, span):
    """
    Decorator function for <code>BertTokens2WordsRewriter</code>. \n
    Aggregates the <code>form</code>, <code>partofspeech</code> and <code>probabilities</code> from <code>shared_bert_tokens</code>, finds the most
    common top-1 label, and retrieves the top N labels and their probabilities from the
    first token that contains this top-1 label.

    Args:
        text_obj: EstNLTK Text object.
        words_index: Index of the word in <code>words</code> layer.
        span: EstNLTK's Span object.

    Returns:
        dict: Annotations with the top N labels and probabilities for the word/phrase.
    """

    # Step 1: Find the most frequent top-1 label across all tokens
    top_1_label_counts = collections.Counter()
    for sp in span:
        forms = sp['form']
        poses = sp['partofspeech']
        for form, pos in zip(forms, poses):
            top_1_label = form + '_' + pos # Get top-1 label
            top_1_label_counts[top_1_label] += 1  # Count occurrences of each top-1 label

    # Identify the most frequent top-1 label
    most_frequent_label = top_1_label_counts.most_common(1)[0][0]
    annotations = list()

    # Step 2: Find the first token that has this most frequent top-1 label
    for sp in span:
        form = most_frequent_label.split('_')[0]
        pos = most_frequent_label.split('_')[1]
        if form in sp['form'] and pos in sp['partofspeech']:

            # Extract the top N labels and their probabilities starting from this label
            tokens = [sp['bert_tokens'][0] for sp in span]
            forms = sp['form']
            poses = sp['partofspeech']
            probabilities = sp['probability']
            for form, pos, probability in zip(forms, poses, probabilities):
                annotation = {
                                'bert_tokens': tokens,
                                'form': form,
                                'partofspeech': pos,
                                'probability': probability
                            }
                annotations.append(annotation)

            # Return the final annotation
            return annotations

    # Fallback if no label found (shouldn't happen)
    raise RuntimeError(f'Could not find a token with this label: {most_frequent_label}')


def _split_sentence_into_smaller_chunks(large_sent: str, max_size:int=900, seek_end_symbols: str='.!?'):
    """
    Splits given large_sent into smaller texts following the text size limit.
    Each smaller text string is allowed to have at most `max_size` characters.
    Returns smaller text strings and their (start, end) indexes in the large_sent.
    """
    assert max_size > 0, f'(!) Invalid batch size: {max_size}'
    if len(large_sent) < max_size:
        return [large_sent], [(0, len(large_sent))]
    chunks = []
    chunk_separators = []
    chunk_indexes = []
    last_chunk_end = 0
    while last_chunk_end < len(large_sent):
        chunk_start = last_chunk_end
        chunk_end = chunk_start + max_size
        if chunk_end >= len(large_sent):
            chunk_end = len(large_sent)
        if isinstance(seek_end_symbols, str):
            # Heuristic: Try to find the last position in the chunk that
            # resembles sentence ending (matches one of the seek_end_symbols)
            i = chunk_end - 1
            while i > chunk_start + 1:
                char = large_sent[i]
                if char in seek_end_symbols:
                    chunk_end = i + 1
                    break
                i -= 1
        chunks.append( large_sent[chunk_start:chunk_end] )
        chunk_indexes.append( (chunk_start, chunk_end) )
        # Find next chunk_start, skip space characters
        updated_chunk_end = chunk_end
        if chunk_end != len(large_sent):
            i = chunk_end
            while i < len(large_sent):
                char = large_sent[i]
                if not char.isspace():
                    updated_chunk_end = i
                    break
                i += 1
            chunk_separators.append( large_sent[chunk_end:updated_chunk_end] )
        last_chunk_end = updated_chunk_end
    assert len(chunk_separators) == len(chunks) - 1
    # Return extracted chunks
    return ( chunks, chunk_indexes )

In [10]:
bert2 = BertMorphTagger2(output_layer ="morph_analysis" , disambiguate=True)

In [13]:
t1 = Text("Välistatud ei ole ka see, et järveni ulatus mingi osa Taani hindamisraamatus nimetatud Väo (Uv� tho) külast. ")
t1.tag_layer(["words", "sentences", "morph_analysis"])
bert2.retag(t1)

Text(text='Välistatud ei ole ka see, et järveni ulatus mingi osa Taani hindamisraamatus nimetatud Väo (Uv� tho) külast. ')

The result will have the span with special symbol.

In [14]:
t1["morph_analysis"]

Layer(name='morph_analysis', attributes=('normalized_text', 'lemma', 'root', 'root_tokens', 'ending', 'clitic', 'form', 'partofspeech'), spans=SL[Span('Välistatud', [{'normalized_text': 'Välistatud', 'lemma': 'välistatu', 'root': 'välista=tu', 'root_tokens': ['välistatu'], 'ending': 'd', 'clitic': '', 'form': 'pl n', 'partofspeech': 'S'}]),
Span('ei', [{'normalized_text': 'ei', 'lemma': 'ei', 'root': 'ei', 'root_tokens': ['ei'], 'ending': '0', 'clitic': '', 'form': 'neg', 'partofspeech': 'V'}]),
Span('ole', [{'normalized_text': 'ole', 'lemma': 'olema', 'root': 'ole', 'root_tokens': ['ole'], 'ending': '0', 'clitic': '', 'form': 'o', 'partofspeech': 'V'}]),
Span('ka', [{'normalized_text': 'ka', 'lemma': 'ka', 'root': 'ka', 'root_tokens': ['ka'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'D'}]),
Span('see', [{'normalized_text': 'see', 'lemma': 'see', 'root': 'see', 'root_tokens': ['see'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'P'}]),
Span(',', [{'normalized_text': ',', 'lemma': ',', 'root': ',', 'root_tokens': [','], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}]),
Span('et', [{'normalized_text': 'et', 'lemma': 'et', 'root': 'et', 'root_tokens': ['et'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'J'}]),
Span('järveni', [{'normalized_text': 'järveni', 'lemma': 'järv', 'root': 'järv', 'root_tokens': ['järv'], 'ending': 'ni', 'clitic': '', 'form': 'sg ter', 'partofspeech': 'S'}]),
Span('ulatus', [{'normalized_text': 'ulatus', 'lemma': 'ulatuma', 'root': 'ulatu', 'root_tokens': ['ulatu'], 'ending': 's', 'clitic': '', 'form': 's', 'partofspeech': 'V'}]),
Span('mingi', [{'normalized_text': 'mingi', 'lemma': 'mingi', 'root': 'mingi', 'root_tokens': ['mingi'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'P'}]),
Span('osa', [{'normalized_text': 'osa', 'lemma': 'osa', 'root': 'osa', 'root_tokens': ['osa'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span('Taani', [{'normalized_text': 'Taani', 'lemma': 'Taani', 'root': 'Taani', 'root_tokens': ['Taani'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeech': 'H'}]),
Span('hindamisraamatus', [{'normalized_text': 'hindamisraamatus', 'lemma': 'hindamisraamat', 'root': 'hindamis_raamat', 'root_tokens': ['hindamis', 'raamat'], 'ending': 's', 'clitic': '', 'form': 'sg in', 'partofspeech': 'S'}]),
Span('nimetatud', [{'normalized_text': 'nimetatud', 'lemma': 'nimetatud', 'root': 'nimetatud', 'root_tokens': ['nimetatud'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'A'}]),
Span('Väo', [{'normalized_text': 'Väo', 'lemma': 'Väo', 'root': 'Väo', 'root_tokens': ['Väo'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeech': 'H'}]),
Span('(', [{'normalized_text': '(', 'lemma': '(', 'root': '(', 'root_tokens': ['('], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}]),
Span('Uv', [{'normalized_text': 'Uv', 'lemma': 'Uv', 'root': 'Uv', 'root_tokens': ['Uv'], 'ending': '0', 'clitic': '', 'form': '?', 'partofspeech': 'Y'}]),
Span('�', [{'normalized_text': '�', 'lemma': '�', 'root': '�', 'root_tokens': ['�'], 'ending': '0', 'clitic': '', 'form': '?', 'partofspeech': 'Y'}]),
Span('tho', [{'normalized_text': 'tho', 'lemma': 'tho', 'root': 'tho', 'root_tokens': ['tho'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span(')', [{'normalized_text': ')', 'lemma': ')', 'root': ')', 'root_tokens': [')'], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}]),
Span('külast', [{'normalized_text': 'külast', 'lemma': 'küla', 'root': 'küla', 'root_tokens': ['küla'], 'ending': 'st', 'clitic': '', 'form': 'sg el', 'partofspeech': 'S'}]),
Span('.', [{'normalized_text': '.', 'lemma': '.', 'root': '.', 'root_tokens': ['.'], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}])])

### BertMorphTagger (disambiguate=False) 

In [15]:
bert3 = BertMorphTagger2(output_layer ="morph_analysis" , disambiguate=False)

In [16]:
t1 = Text("Välistatud ei ole ka see, et järveni ulatus mingi osa Taani hindamisraamatus nimetatud Väo (Uv� tho) külast. ")
t1.tag_layer(["words", "sentences"])
bert3.tag(t1)

Text(text='Välistatud ei ole ka see, et järveni ulatus mingi osa Taani hindamisraamatus nimetatud Väo (Uv� tho) külast. ')

**The result will have the span with special symbol and empty attributes.**

In [17]:
t1["morph_analysis"]

Layer(name='morph_analysis', attributes=('bert_tokens', 'form', 'partofspeech', 'probability'), spans=SL[Span('Välistatud', [{'bert_tokens': ['▁Vä', 'list', 'atud'], 'form': 'tud', 'partofspeech': 'V', 'probability': 0.99557}]),
Span('ei', [{'bert_tokens': ['▁ei'], 'form': 'neg', 'partofspeech': 'V', 'probability': 0.99996}]),
Span('ole', [{'bert_tokens': ['▁ole'], 'form': 'neg o', 'partofspeech': 'V', 'probability': 0.99992}]),
Span('ka', [{'bert_tokens': ['▁ka'], 'form': '', 'partofspeech': 'D', 'probability': 0.99996}]),
Span('see', [{'bert_tokens': ['▁see'], 'form': 'sg n', 'partofspeech': 'P', 'probability': 0.99993}]),
Span(',', [{'bert_tokens': [','], 'form': '', 'partofspeech': 'Z', 'probability': 0.99997}]),
Span('et', [{'bert_tokens': ['▁et'], 'form': '', 'partofspeech': 'J', 'probability': 0.99994}]),
Span('järveni', [{'bert_tokens': ['▁järv', 'eni'], 'form': 'sg ter', 'partofspeech': 'S', 'probability': 0.99867}]),
Span('ulatus', [{'bert_tokens': ['▁ulatus'], 'form': 's', 'partofspeech': 'V', 'probability': 0.99986}]),
Span('mingi', [{'bert_tokens': ['▁mingi'], 'form': 'sg n', 'partofspeech': 'P', 'probability': 0.99992}]),
Span('osa', [{'bert_tokens': ['▁osa'], 'form': 'sg n', 'partofspeech': 'S', 'probability': 0.9994}]),
Span('Taani', [{'bert_tokens': ['▁Taani'], 'form': 'sg g', 'partofspeech': 'H', 'probability': 0.99921}]),
Span('hindamisraamatus', [{'bert_tokens': ['▁hindamis', 'raamatus'], 'form': 'sg in', 'partofspeech': 'S', 'probability': 0.9998}]),
Span('nimetatud', [{'bert_tokens': ['▁nimetatud'], 'form': '', 'partofspeech': 'A', 'probability': 0.9998}]),
Span('Väo', [{'bert_tokens': ['▁Vä', 'o'], 'form': 'sg g', 'partofspeech': 'H', 'probability': 0.99902}]),
Span('(', [{'bert_tokens': ['▁('], 'form': '', 'partofspeech': 'Z', 'probability': 0.99989}]),
Span('Uv', [{'bert_tokens': ['U', 'v'], 'form': '', 'partofspeech': 'Y', 'probability': 0.70038}]),
Span('�', [{'bert_tokens': [None], 'form': '', 'partofspeech': '', 'probability': None}]),
Span('tho', [{'bert_tokens': ['▁t', 'ho'], 'form': '', 'partofspeech': 'Y', 'probability': 0.8381}]),
Span(')', [{'bert_tokens': [')'], 'form': '', 'partofspeech': 'Z', 'probability': 0.99996}]),
Span('külast', [{'bert_tokens': ['▁külast'], 'form': 'sg el', 'partofspeech': 'S', 'probability': 0.99973}]),
Span('.', [{'bert_tokens': ['.'], 'form': '', 'partofspeech': 'Z', 'probability': 0.99998}])])

## More examples, where there are more special symbols at different locations

In [18]:
t2 = Text("Tema F2 kaugus eesvokaal � F2-st on 1,17 barki ja eesvokaal � F2-st 0,57 barki ning tagavokaal � F2-st 3,96 barki ja tagavokaal � F2-st 1,18 barki .")
t2.tag_layer(["words", "sentences", "morph_analysis"])
bert2.retag( t2 )

Text(text='Tema F2 kaugus eesvokaal � F2-st on 1,17 barki ja eesvokaal � F2-st 0,57 barki ning tagavokaal � F2-st 3,96 barki ja tagavokaal � F2-st 1,18 barki .')

In [19]:
t2["morph_analysis"]

Layer(name='morph_analysis', attributes=('normalized_text', 'lemma', 'root', 'root_tokens', 'ending', 'clitic', 'form', 'partofspeech'), spans=SL[Span('Tema', [{'normalized_text': 'Tema', 'lemma': 'tema', 'root': 'tema', 'root_tokens': ['tema'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeech': 'P'}]),
Span('F2', [{'normalized_text': 'F2', 'lemma': 'F2', 'root': 'F2', 'root_tokens': ['F2'], 'ending': '0', 'clitic': '', 'form': '?', 'partofspeech': 'Y'}]),
Span('kaugus', [{'normalized_text': 'kaugus', 'lemma': 'kaugus', 'root': 'kaugus', 'root_tokens': ['kaugus'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span('eesvokaal', [{'normalized_text': 'eesvokaal', 'lemma': 'eesvokaal', 'root': 'ees_vokaal', 'root_tokens': ['ees', 'vokaal'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span('�', [{'normalized_text': '�', 'lemma': '�', 'root': '�', 'root_tokens': ['�'], 'ending': '0', 'clitic': '', 'form': '?', 'partofspeech': 'Y'}]),
Span('F2-st', [{'normalized_text': 'F2-st', 'lemma': 'F2', 'root': 'F2', 'root_tokens': ['F2'], 'ending': 'st', 'clitic': '', 'form': 'sg el', 'partofspeech': 'Y'}]),
Span('on', [{'normalized_text': 'on', 'lemma': 'olema', 'root': 'ole', 'root_tokens': ['ole'], 'ending': '0', 'clitic': '', 'form': 'b', 'partofspeech': 'V'}]),
Span('1,17', [{'normalized_text': '1,17', 'lemma': '1,17', 'root': '1,17', 'root_tokens': ['1,17'], 'ending': '0', 'clitic': '', 'form': '?', 'partofspeech': 'N'}]),
Span('barki', [{'normalized_text': 'barki', 'lemma': 'bark', 'root': 'bark', 'root_tokens': ['bark'], 'ending': '0', 'clitic': '', 'form': 'sg p', 'partofspeech': 'S'}]),
Span('ja', [{'normalized_text': 'ja', 'lemma': 'ja', 'root': 'ja', 'root_tokens': ['ja'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'J'}]),
Span('eesvokaal', [{'normalized_text': 'eesvokaal', 'lemma': 'eesvokaal', 'root': 'ees_vokaal', 'root_tokens': ['ees', 'vokaal'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span('�', [{'normalized_text': '�', 'lemma': '�', 'root': '�', 'root_tokens': ['�'], 'ending': '0', 'clitic': '', 'form': '?', 'partofspeech': 'Y'}]),
Span('F2-st', [{'normalized_text': 'F2-st', 'lemma': 'F2', 'root': 'F2', 'root_tokens': ['F2'], 'ending': 'st', 'clitic': '', 'form': 'sg el', 'partofspeech': 'Y'}]),
Span('0,57', [{'normalized_text': '0,57', 'lemma': '0,57', 'root': '0,57', 'root_tokens': ['0,57'], 'ending': '0', 'clitic': '', 'form': '?', 'partofspeech': 'N'}]),
Span('barki', [{'normalized_text': 'barki', 'lemma': 'barki', 'root': 'barki', 'root_tokens': ['barki'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span('ning', [{'normalized_text': 'ning', 'lemma': 'ning', 'root': 'ning', 'root_tokens': ['ning'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'J'}]),
Span('tagavokaal', [{'normalized_text': 'tagavokaal', 'lemma': 'tagavokaal', 'root': 'taga_vokaal', 'root_tokens': ['taga', 'vokaal'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span('�', [{'normalized_text': '�', 'lemma': '�', 'root': '�', 'root_tokens': ['�'], 'ending': '0', 'clitic': '', 'form': '?', 'partofspeech': 'Y'}]),
Span('F2-st', [{'normalized_text': 'F2-st', 'lemma': 'F2', 'root': 'F2', 'root_tokens': ['F2'], 'ending': 'st', 'clitic': '', 'form': 'sg el', 'partofspeech': 'Y'}]),
Span('3,96', [{'normalized_text': '3,96', 'lemma': '3,96', 'root': '3,96', 'root_tokens': ['3,96'], 'ending': '0', 'clitic': '', 'form': '?', 'partofspeech': 'N'}]),
Span('barki', [{'normalized_text': 'barki', 'lemma': 'barki', 'root': 'barki', 'root_tokens': ['barki'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span('ja', [{'normalized_text': 'ja', 'lemma': 'ja', 'root': 'ja', 'root_tokens': ['ja'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'J'}]),
Span('tagavokaal', [{'normalized_text': 'tagavokaal', 'lemma': 'tagavokaal', 'root': 'taga_vokaal', 'root_tokens': ['taga', 'vokaal'], 'endin

In [20]:
t3 = Text("� - mootori võlli pöördenurk , Tem - elektromagnetiline pöördemoment , Ts - koormuse pöördemoment , J - inertsimoment . ")
t3.tag_layer(["words", "sentences", "morph_analysis"])
bert2.retag( t3 )

/home/kaire/anaconda3/envs/base2/lib/python3.10/site-packages/estnltk_neural/taggers/embeddings/bert/bert_tokens_to_words_rewriter.py:192: UserWarning: (!) No matching words span for bert token Span(' ', [{'bert_tokens': '▁', 'form': 'sg n', 'partofspeech': 'S', 'probability': 0.9999}]).
  warnings.warn(f"(!) No matching {words_layer.name} span for bert token {bert_tokens_layer[i]}.")


Text(text='� - mootori võlli pöördenurk , Tem - elektromagnetiline pöördemoment , Ts - koormuse pöördemoment , J - inertsimoment . ')

In [21]:
t3["morph_analysis"]

Layer(name='morph_analysis', attributes=('normalized_text', 'lemma', 'root', 'root_tokens', 'ending', 'clitic', 'form', 'partofspeech'), spans=SL[Span('�', [{'normalized_text': '�', 'lemma': '�', 'root': '�', 'root_tokens': ['�'], 'ending': '0', 'clitic': '', 'form': '?', 'partofspeech': 'Y'}]),
Span('-', [{'normalized_text': '-', 'lemma': '-', 'root': '-', 'root_tokens': ['-'], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}]),
Span('mootori', [{'normalized_text': 'mootori', 'lemma': 'mootor', 'root': 'mootor', 'root_tokens': ['mootor'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeech': 'S'}]),
Span('võlli', [{'normalized_text': 'võlli', 'lemma': 'võll', 'root': 'võll', 'root_tokens': ['võll'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeech': 'S'}]),
Span('pöördenurk', [{'normalized_text': 'pöördenurk', 'lemma': 'pöördenurk', 'root': 'pöörde_nurk', 'root_tokens': ['pöörde', 'nurk'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span(',', [{'normalized_text': ',', 'lemma': ',', 'root': ',', 'root_tokens': [','], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}]),
Span('Tem', [{'normalized_text': 'Tem', 'lemma': 'Tem', 'root': 'Tem', 'root_tokens': ['Tem'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'H'}]),
Span('-', [{'normalized_text': '-', 'lemma': '-', 'root': '-', 'root_tokens': ['-'], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}]),
Span('elektromagnetiline', [{'normalized_text': 'elektromagnetiline', 'lemma': 'elektromagnetiline', 'root': 'elektro_magneti=line', 'root_tokens': ['elektro', 'magnetiline'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'A'}]),
Span('pöördemoment', [{'normalized_text': 'pöördemoment', 'lemma': 'pöördemoment', 'root': 'pöörde_moment', 'root_tokens': ['pöörde', 'moment'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span(',', [{'normalized_text': ',', 'lemma': ',', 'root': ',', 'root_tokens': [','], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}]),
Span('Ts', [{'normalized_text': 'Ts', 'lemma': 'Ts', 'root': 'Ts', 'root_tokens': ['Ts'], 'ending': '0', 'clitic': '', 'form': '?', 'partofspeech': 'Y'}]),
Span('-', [{'normalized_text': '-', 'lemma': '-', 'root': '-', 'root_tokens': ['-'], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}]),
Span('koormuse', [{'normalized_text': 'koormuse', 'lemma': 'koormus', 'root': 'koormus', 'root_tokens': ['koormus'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeech': 'S'}]),
Span('pöördemoment', [{'normalized_text': 'pöördemoment', 'lemma': 'pöördemoment', 'root': 'pöörde_moment', 'root_tokens': ['pöörde', 'moment'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span(',', [{'normalized_text': ',', 'lemma': ',', 'root': ',', 'root_tokens': [','], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}]),
Span('J', [{'normalized_text': 'J', 'lemma': 'J', 'root': 'J', 'root_tokens': ['J'], 'ending': '0', 'clitic': '', 'form': '?', 'partofspeech': 'Y'}]),
Span('-', [{'normalized_text': '-', 'lemma': '-', 'root': '-', 'root_tokens': ['-'], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}]),
Span('inertsimoment', [{'normalized_text': 'inertsimoment', 'lemma': 'inertsimoment', 'root': 'inertsi_moment', 'root_tokens': ['inertsi', 'moment'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span('.', [{'normalized_text': '.', 'lemma': '.', 'root': '.', 'root_tokens': ['.'], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}])])

In [22]:
t4 = Text(" Ülipika kõrge�harva esinemuse tõttu asendatakse käesolevas töös tavaliselt ka kõrgenenud keskkõrge vokaal v� kõrge vokaaliga �")
t4.tag_layer(["words", "sentences", "morph_analysis"])
bert2.retag( t4 )

Text(text=' Ülipika kõrge�harva esinemuse tõttu asendatakse käesolevas töös tavaliselt ka kõrgenenud keskkõrge vokaal v� kõrge vokaaliga �')

In [23]:
t4["morph_analysis"]

Layer(name='morph_analysis', attributes=('normalized_text', 'lemma', 'root', 'root_tokens', 'ending', 'clitic', 'form', 'partofspeech'), spans=SL[Span('Ülipika', [{'normalized_text': 'Ülipika', 'lemma': 'Ülipikk', 'root': 'Üli_pikk', 'root_tokens': ['Üli', 'pikk'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeech': 'H'}, {'normalized_text': 'Ülipika', 'lemma': 'Ülipika', 'root': 'Ülipika', 'root_tokens': ['Ülipika'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeech': 'H'}, {'normalized_text': 'Ülipika', 'lemma': 'Ülipikas', 'root': 'Ülipikas', 'root_tokens': ['Ülipikas'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeech': 'H'}]),
Span('kõrge', [{'normalized_text': 'kõrge', 'lemma': 'kõrge', 'root': 'kõrge', 'root_tokens': ['kõrge'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'A'}]),
Span('�', [{'normalized_text': '�', 'lemma': '�', 'root': '�', 'root_tokens': ['�'], 'ending': '0', 'clitic': '', 'form': '?', 'partofspeech': 'Y'}]),
Span('harva', [{'normalized_text': 'harva', 'lemma': 'harva', 'root': 'harva', 'root_tokens': ['harva'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'D'}]),
Span('esinemuse', [{'normalized_text': 'esinemuse', 'lemma': 'esinemus', 'root': 'esinemus', 'root_tokens': ['esinemus'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeech': 'S'}]),
Span('tõttu', [{'normalized_text': 'tõttu', 'lemma': 'tõttu', 'root': 'tõttu', 'root_tokens': ['tõttu'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'K'}]),
Span('asendatakse', [{'normalized_text': 'asendatakse', 'lemma': 'asendama', 'root': 'asenda', 'root_tokens': ['asenda'], 'ending': 'takse', 'clitic': '', 'form': 'takse', 'partofspeech': 'V'}]),
Span('käesolevas', [{'normalized_text': 'käesolevas', 'lemma': 'käesolev', 'root': 'käes_olev', 'root_tokens': ['käes', 'olev'], 'ending': 's', 'clitic': '', 'form': 'sg in', 'partofspeech': 'A'}]),
Span('töös', [{'normalized_text': 'töös', 'lemma': 'töö', 'root': 'töö', 'root_tokens': ['töö'], 'ending': 's', 'clitic': '', 'form': 'sg in', 'partofspeech': 'S'}]),
Span('tavaliselt', [{'normalized_text': 'tavaliselt', 'lemma': 'tavaliselt', 'root': 'tavaliselt', 'root_tokens': ['tavaliselt'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'D'}]),
Span('ka', [{'normalized_text': 'ka', 'lemma': 'ka', 'root': 'ka', 'root_tokens': ['ka'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'D'}]),
Span('kõrgenenud', [{'normalized_text': 'kõrgenenud', 'lemma': 'kõrgenenud', 'root': 'kõrgene=nud', 'root_tokens': ['kõrgenenud'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'A'}]),
Span('keskkõrge', [{'normalized_text': 'keskkõrge', 'lemma': 'keskkõrge', 'root': 'kesk_kõrge', 'root_tokens': ['kesk', 'kõrge'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'A'}]),
Span('vokaal', [{'normalized_text': 'vokaal', 'lemma': 'vokaal', 'root': 'vokaal', 'root_tokens': ['vokaal'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span('v', [{'normalized_text': 'v', 'lemma': 'v', 'root': 'v', 'root_tokens': ['v'], 'ending': '0', 'clitic': '', 'form': '?', 'partofspeech': 'Y'}]),
Span('�', [{'normalized_text': '�', 'lemma': '�', 'root': '�', 'root_tokens': ['�'], 'ending': '0', 'clitic': '', 'form': '?', 'partofspeech': 'Y'}]),
Span('kõrge', [{'normalized_text': 'kõrge', 'lemma': 'kõrge', 'root': 'kõrge', 'root_tokens': ['kõrge'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeech': 'A'}]),
Span('vokaaliga', [{'normalized_text': 'vokaaliga', 'lemma': 'vokaal', 'root': 'vokaal', 'root_tokens': ['vokaal'], 'ending': 'ga', 'clitic': '', 'form': 'sg kom', 'partofspeech': 'S'}]),
Span('�', [{'normalized_text': '�', 'lemma': '�', 'root': '�', 'root_tokens': ['�'], 'ending': '0', 'clitic': '', 'form': '?', 'partofspeech': 'Y'}])])

In [24]:
t5 = Text("� ��")
t5.tag_layer(["words", "sentences", "morph_analysis"])
bert2.retag( t5 )

Text(text='� ��')

In [25]:
t5["morph_analysis"]

Layer(name='morph_analysis', attributes=('normalized_text', 'lemma', 'root', 'root_tokens', 'ending', 'clitic', 'form', 'partofspeech'), spans=SL[Span('�', [{'normalized_text': '�', 'lemma': '�', 'root': '�', 'root_tokens': ['�'], 'ending': '0', 'clitic': '', 'form': '?', 'partofspeech': 'Y'}]),
Span('��', [{'normalized_text': '��', 'lemma': '��', 'root': '��', 'root_tokens': ['��'], 'ending': '0', 'clitic': '', 'form': '?', 'partofspeech': 'Y'}])])

In [26]:
# still works for normal text
t6 = Text("Välistatud ei ole ka see, et järveni ulatus mingi osa Taani hindamisraamatus nimetatud Väo (Uv tho) külast. ")
t6.tag_layer(["words", "sentences", "morph_analysis"])
bert2.retag( t6 )

Text(text='Välistatud ei ole ka see, et järveni ulatus mingi osa Taani hindamisraamatus nimetatud Väo (Uv tho) külast. ')

In [27]:
t6["morph_analysis"]

Layer(name='morph_analysis', attributes=('normalized_text', 'lemma', 'root', 'root_tokens', 'ending', 'clitic', 'form', 'partofspeech'), spans=SL[Span('Välistatud', [{'normalized_text': 'Välistatud', 'lemma': 'välistatu', 'root': 'välista=tu', 'root_tokens': ['välistatu'], 'ending': 'd', 'clitic': '', 'form': 'pl n', 'partofspeech': 'S'}]),
Span('ei', [{'normalized_text': 'ei', 'lemma': 'ei', 'root': 'ei', 'root_tokens': ['ei'], 'ending': '0', 'clitic': '', 'form': 'neg', 'partofspeech': 'V'}]),
Span('ole', [{'normalized_text': 'ole', 'lemma': 'olema', 'root': 'ole', 'root_tokens': ['ole'], 'ending': '0', 'clitic': '', 'form': 'o', 'partofspeech': 'V'}]),
Span('ka', [{'normalized_text': 'ka', 'lemma': 'ka', 'root': 'ka', 'root_tokens': ['ka'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'D'}]),
Span('see', [{'normalized_text': 'see', 'lemma': 'see', 'root': 'see', 'root_tokens': ['see'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'P'}]),
Span(',', [{'normalized_text': ',', 'lemma': ',', 'root': ',', 'root_tokens': [','], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}]),
Span('et', [{'normalized_text': 'et', 'lemma': 'et', 'root': 'et', 'root_tokens': ['et'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'J'}]),
Span('järveni', [{'normalized_text': 'järveni', 'lemma': 'järv', 'root': 'järv', 'root_tokens': ['järv'], 'ending': 'ni', 'clitic': '', 'form': 'sg ter', 'partofspeech': 'S'}]),
Span('ulatus', [{'normalized_text': 'ulatus', 'lemma': 'ulatuma', 'root': 'ulatu', 'root_tokens': ['ulatu'], 'ending': 's', 'clitic': '', 'form': 's', 'partofspeech': 'V'}]),
Span('mingi', [{'normalized_text': 'mingi', 'lemma': 'mingi', 'root': 'mingi', 'root_tokens': ['mingi'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'P'}]),
Span('osa', [{'normalized_text': 'osa', 'lemma': 'osa', 'root': 'osa', 'root_tokens': ['osa'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span('Taani', [{'normalized_text': 'Taani', 'lemma': 'Taani', 'root': 'Taani', 'root_tokens': ['Taani'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeech': 'H'}]),
Span('hindamisraamatus', [{'normalized_text': 'hindamisraamatus', 'lemma': 'hindamisraamat', 'root': 'hindamis_raamat', 'root_tokens': ['hindamis', 'raamat'], 'ending': 's', 'clitic': '', 'form': 'sg in', 'partofspeech': 'S'}]),
Span('nimetatud', [{'normalized_text': 'nimetatud', 'lemma': 'nimetatud', 'root': 'nimetatud', 'root_tokens': ['nimetatud'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'A'}]),
Span('Väo', [{'normalized_text': 'Väo', 'lemma': 'Väo', 'root': 'Väo', 'root_tokens': ['Väo'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeech': 'H'}]),
Span('(', [{'normalized_text': '(', 'lemma': '(', 'root': '(', 'root_tokens': ['('], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}]),
Span('Uv', [{'normalized_text': 'Uv', 'lemma': 'Uv', 'root': 'Uv', 'root_tokens': ['Uv'], 'ending': '0', 'clitic': '', 'form': '?', 'partofspeech': 'Y'}]),
Span('tho', [{'normalized_text': 'tho', 'lemma': 'tho', 'root': 'tho', 'root_tokens': ['tho'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span(')', [{'normalized_text': ')', 'lemma': ')', 'root': ')', 'root_tokens': [')'], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}]),
Span('külast', [{'normalized_text': 'külast', 'lemma': 'küla', 'root': 'küla', 'root_tokens': ['küla'], 'ending': 'st', 'clitic': '', 'form': 'sg el', 'partofspeech': 'S'}]),
Span('.', [{'normalized_text': '.', 'lemma': '.', 'root': '.', 'root_tokens': ['.'], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}])])